# Taller — Limpieza y Análisis de Datos con Pandas y NumPy

**Objetivo:** Aplicar técnicas de limpieza, transformación y análisis de datos sobre un dataset con inconsistencias reales.

In [69]:
# Importación de librerías
import pandas as pd
import numpy as np

---
## Parte 1 — Exploración de Datos

In [70]:
# Cargar el dataset
df = pd.read_csv('ventas_sucias_5000.csv')

# Hacemos una breve visualización de las primeras filas.
print("Las primeras 5 filas del dataset son: ")
df.head()

Las primeras 5 filas del dataset son: 


,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
0,Maria,Monitor,1326.0,NaN,peru,Efectivo,2024-01-01 00:00:00
1,Luisa,Laptop,55.0,2,chile,Tarjeta,2024-01-01 01:00:00
2,Carlos,Monitor,1203.0,9,Colombia,Efectivo,2024-01-01 02:00:00
3,Luisa,Monitor,1304.0,3,Perú,TRANSFERENCIA,2024-01-01 03:00:00
4,Luisa,Monitor,426.0,6,chile,Tarjeta,2024-01-01 04:00:00


In [71]:
# Información general del dataset
print("Información general: ")
df.info()

Información general: 
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente      5000 non-null   str    
 1   producto     5000 non-null   str    
 2   precio       4950 non-null   float64
 3   cantidad     4950 non-null   str    
 4   pais         5000 non-null   str    
 5   metodo_pago  5000 non-null   str    
 6   fecha        5000 non-null   str    
dtypes: float64(1), str(6)
memory usage: 273.6 KB


In [72]:
# Resumen estadístico
print("Resumen estadístico")
df.describe(include = "all")

Resumen estadístico


,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
count,5000,5000,4950.000000,4950,5000,5000,5000
unique,6,5,NaN,10,7,4,4825
top,Ana,Mouse,NaN,2,chile,TRANSFERENCIA,2024-01-19 00:00:00
freq,870,1069,NaN,573,733,1276,5
mean,NaN,NaN,5053.823434,NaN,NaN,NaN,NaN
std,NaN,NaN,63379.982009,NaN,NaN,NaN,NaN
min,NaN,NaN,10.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,531.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,1027.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,1519.000000,NaN,NaN,NaN,NaN


In [73]:
# Vemos cuál es el tamaño del DataFrame

df.isnull().sum()

cliente         0
producto        0
precio         50
cantidad       50
pais            0
metodo_pago     0
fecha           0
dtype: int64

### Respuestas de la exploración

**¿Cuántas filas y columnas tiene la base de datos?**

In [74]:
filas, columnas = df.shape
print(f"Filas: {filas}")
print(f"Columnas: {columnas}")
# El dataset tiene 5.000 registros y 7 columnas.

Filas: 5000
Columnas: 7


**¿Qué tipos de datos identificas?**

In [75]:
print(df.dtypes)
# - cliente, producto, pais, metodo_pago, fecha → tipo object (texto)
# - precio → float64 (numérico con decimales)
# - cantidad → object (debería ser numérico, pero viene como texto)
# - 'fecha' debería ser datetime en vez de obj o str.
# El tipo de 'cantidad' es incorrecto.

cliente            str
producto           str
precio         float64
cantidad           str
pais               str
metodo_pago        str
fecha              str
dtype: object


**¿Encuentras columnas con problemas o inconsistencias?**

In [76]:
# Valores nulos por columna
print("Valores nulos")
print(df.isnull().sum())

# Valores únicos en columnas categóricas
print("\nValores únicos en 'pais'")
print(df['pais'].unique())

print("\nValores únicos en 'metodo_pago'")
print(df['metodo_pago'].unique())

print("\nPrecios máximos sospechosamente incoherentes")
print(df['precio'].sort_values(ascending=False).head(10))

print("\nValores no numéricos en 'cantidad'")
cant_num = pd.to_numeric(df['cantidad'], errors='coerce')
mask_texto = cant_num.isnull() & df['cantidad'].notna()
print(df['cantidad'][mask_texto].unique())

# PROBLEMAS IDENTIFICADOS:
# 1. 'precio' y 'cantidad': 50 valores nulos cada uno
# 2. 'cantidad': tipo incorrecto (str); contiene la palabra 'three' (texto en vez de número)
# 3. 'pais': inconsistencias de capitalización y abreviaciones
#    ('peru', 'Perú', 'chile', 'Chile', 'Colombia', 'COL', 'col')
# 4. 'metodo_pago': capitalización inconsistente
#    ('Efectivo', 'Tarjeta', 'TRANSFERENCIA', 'transferencia')
# 5. 'precio': 20 registros con valor 999999 (probable valor centinela/error)
# 6. 'fecha': tipo str, debe convertirse a datetime

Valores nulos
cliente         0
producto        0
precio         50
cantidad       50
pais            0
metodo_pago     0
fecha           0
dtype: int64

Valores únicos en 'pais'
<StringArray>
['peru', 'chile', 'Colombia', 'Perú', 'COL', 'Chile', 'col']
Length: 7, dtype: str

Valores únicos en 'metodo_pago'
<StringArray>
['Efectivo', 'Tarjeta', 'TRANSFERENCIA', 'transferencia']
Length: 4, dtype: str

Precios máximos sospechosamente incoherentes
771     999999.0
2825    999999.0
1165    999999.0
1159    999999.0
3868    999999.0
1885    999999.0
2077    999999.0
1123    999999.0
1242    999999.0
2976    999999.0
Name: precio, dtype: float64

Valores no numéricos en 'cantidad'
<StringArray>
['three']
Length: 1, dtype: str


---
## Parte 2 — Limpieza de Datos

In [77]:
# Trabajamos sobre una copia para preservar el original
df_clean = df.copy()

In [78]:
# Aquí, corregí 'cantidad', reemplacé texto 'three' por '3' y convertí a numérico 
df_clean['cantidad'] = df_clean['cantidad'].replace('three', '3')
df_clean['cantidad'] = pd.to_numeric(df_clean['cantidad'], errors='coerce')

print("Tipo de 'cantidad' después de corrección:", df_clean['cantidad'].dtype)
print("Nulos en 'cantidad':", df_clean['cantidad'].isnull().sum())

# La columna tenía 50 nulos originales y valores con texto 'three'.
# Se reemplazó 'three' por su equivalente numérico 3 antes de la conversión.

Tipo de 'cantidad' después de corrección: float64
Nulos en 'cantidad': 50


In [79]:
# Aquí lo que hice fue corregir tipo de 'fecha' a datetime
df_clean['fecha'] = pd.to_datetime(df_clean['fecha'], errors='coerce')

print("Tipo de 'fecha' después de conversión:", df_clean['fecha'].dtype)
print("Nulos en 'fecha':", df_clean['fecha'].isnull().sum())

# La columna estaba almacenada como string; se convierte a datetime64
# para poder hacer operaciones temporales correctamente.

Tipo de 'fecha' después de conversión: datetime64[us]
Nulos en 'fecha': 0


In [80]:
# Estandaricé columna 'pais' 
mapeo_pais = {
    'peru'    : 'Peru',
    'Perú'    : 'Peru',
    'chile'   : 'Chile',
    'Chile'   : 'Chile',
    'Colombia': 'Colombia',
    'COL'     : 'Colombia',
    'col'     : 'Colombia'
}
df_clean['pais'] = df_clean['pais'].map(mapeo_pais)

print("Valores únicos en 'pais' luego de estandarizar:")
print(df_clean['pais'].unique())
# Todos las variantes de strings para representar el mismo dato se unificaron.
# Fue posible eliminando diferencias de mayúsculas, tildes y abreviaciones.

Valores únicos en 'pais' luego de estandarizar:
<StringArray>
['Peru', 'Chile', 'Colombia']
Length: 3, dtype: str


In [81]:
# Estandaricé columna 'metodo_pago' 
df_clean['metodo_pago'] = df_clean['metodo_pago'].str.strip().str.title()

print("Valores únicos en 'metodo_pago' luego de estandarizar:")
print(df_clean['metodo_pago'].unique())
# str.title() convierte cada palabra a primera letra mayúscula,
# unificando 'TRANSFERENCIA' y 'transferencia' → 'Transferencia'.

Valores únicos en 'metodo_pago' luego de estandarizar:
<StringArray>
['Efectivo', 'Tarjeta', 'Transferencia']
Length: 3, dtype: str


In [82]:
# Manejé valores nulos en 'precio' y 'cantidad':

print("Nulos antes:", df_clean[['precio','cantidad']].isnull().sum().to_dict())

# Se eliminan filas donde precio o cantidad son nulos,
# ya que sin ambas columnas no se puede calcular el total de venta.

df_clean = df_clean.dropna(subset=['precio', 'cantidad'])

print("Filas después de eliminar nulos:", len(df_clean))

# Se eliminaron 117 filas (50 de precio + 50 de cantidad originales,
# más el solapamiento con 'three' ya reemplazado).

Nulos antes: {'precio': 50, 'cantidad': 50}
Filas después de eliminar nulos: 4903


In [83]:
# --- 6. Detectar y eliminar outliers en 'precio' ---
print("Registros con precio == 999999:", (df_clean['precio'] == 999999).sum())

# El valor 999999 aparece exactamente 20 veces y no corresponde a
# ningún producto real del catálogo; es un valor centinela típico de
# sistemas que usan ese número para indicar 'dato no disponible'.
df_clean = df_clean[df_clean['precio'] != 999999]

print("Filas después de eliminar outliers:", len(df_clean))
print("Precio máximo tras limpieza:", df_clean['precio'].max())

Registros con precio == 999999: 20
Filas después de eliminar outliers: 4883
Precio máximo tras limpieza: 1997.0


### Respuestas — Parte 2

| Problema | Solución | ¿Se eliminaron registros? |
|---|---|---|
| cantidad` era tipo string | Conversión a numérico con `pd.to_numeric()` | No directamente |
| Texto `'three'` en `cantidad` | Reemplazo previo con `.replace('three','3')` | No |
| 50 nulos en `precio` y `cantidad` | Eliminación de filas con `dropna()` | **Sí — 117 filas** |
| `precio == 999999` (valor centinela) | Filtrado y eliminación | **Sí — 20 filas** |
| `pais` con variantes inconsistentes | Mapeo a valor canónico | No |
| `metodo_pago` con capitalización mixta | `.str.title()` | No |
| `fecha` como string | Conversión con `pd.to_datetime()` | No |

> **Decisión de eliminación:** Se eliminaron filas con nulos en `precio` o `cantidad` porque sin estos valores no es posible calcular el total de venta, que es el indicador central del análisis. Para `precio == 999999`, el valor es claramente un error de carga y distorsionaría gravemente las estadísticas.

---
## Parte 3 — Análisis con Pandas

In [84]:
# Crear columna 'total' = precio × cantidad
df_clean['total'] = df_clean['precio'] * df_clean['cantidad']

print("Columna 'total' creada. Primeras filas:")
df_clean[['producto', 'precio', 'cantidad', 'total']].head()

Columna 'total' creada. Primeras filas:


,producto,precio,cantidad,total
1,Laptop,55.0,2.0,110.0
2,Monitor,1203.0,9.0,10827.0
3,Monitor,1304.0,3.0,3912.0
4,Monitor,426.0,6.0,2556.0
5,Laptop,857.0,6.0,5142.0


In [85]:
# Métricas generales de ventas
total_vendido  = df_clean['total'].sum()
promedio_ventas = df_clean['total'].mean()
venta_maxima   = df_clean['total'].max()
venta_minima   = df_clean['total'].min()

print(f"Total vendido:    $ {total_vendido:,.2f}")
print(f"Promedio ventas:  $ {promedio_ventas:,.2f}")
print(f"Venta máxima:     $ {venta_maxima:,.2f}")
print(f"Venta mínima:     $ {venta_minima:,.2f}")

Total vendido:    $ 24,477,978.00
Promedio ventas:  $ 5,012.90
Venta máxima:     $ 17,955.00
Venta mínima:     $ 10.00


In [86]:
# Top 5 productos por valor total vendido
top5_productos = (
    df_clean.groupby('producto')['total']
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
print("=== Top 5 Productos por Total Vendido ===")
print(top5_productos.apply(lambda x: f"$ {x:,.2f}"))
# Mouse lidera las ventas totales, seguido de Laptop y Monitor.

=== Top 5 Productos por Total Vendido ===
producto
Mouse      $ 5,282,469.00
Laptop     $ 4,966,821.00
Monitor    $ 4,834,497.00
Celular    $ 4,781,259.00
Teclado    $ 4,612,932.00
Name: total, dtype: str


In [ ]:
# País con más ventas totales
ventas_por_pais = (
    df_clean.groupby('pais')['total']
    .sum()
    .sort_values(ascending=False)
)
print("Ventas por País")
print(ventas_por_pais.apply(lambda x: f"$ {x:,.2f}"))
print(f"\nPaís con más ventas: {ventas_por_pais.idxmax()}")
# Colombia lidera el volumen de ventas totales entre los tres países.

=== Ventas por País ===
pais
Colombia    $ 10,568,326.00
Chile        $ 7,033,285.00
Peru         $ 6,876,367.00
Name: total, dtype: str

País con más ventas: Colombia


---
## Parte 4 — Introducción a NumPy

In [94]:
# Convertir columnas numéricas a una matriz de NumPy
data = df_clean[['precio', 'cantidad']].to_numpy()

print("Tipo:", type(data))
print("Shape:", data.shape)
print("Primeras filas del array:")
print(data[:5])
# Se obtiene una matriz 2D de forma (N, 2) donde
# la columna 0 = precios y la columna 1 = cantidades.

Tipo: <class 'numpy.ndarray'>
Shape: (4883, 2)
Primeras filas del array:
[[  55.    2.]
 [1203.    9.]
 [1304.    3.]
 [ 426.    6.]
 [ 857.    6.]]


In [96]:
# Separar columnas usando indexación de NumPy
precios    = data[:, 0]   # Todas las filas, columna 0 → precios
cantidades = data[:, 1]   # Todas las filas, columna 1 → cantidades

print("Primeros precios:   ", precios[:5])
print("Primeras cantidades:", cantidades[:5])

# data[:, 0] usa el slicing propio de NumPy:
#   ':' = todas las filas
#   '0' = columna índice 0
# Esto devuelve una matriz 1D con todos los precios.

Primeros precios:    [  55. 1203. 1304.  426.  857.]
Primeras cantidades: [2. 9. 3. 6. 6.]


In [98]:
# Calcular ventas totales mediante vectorización
totales = precios * cantidades

print("Primeros totales calculados:", totales[:5])
print("Shape de la matriz de totales:", totales.shape)

# La multiplicación opera elemento a elemento sobre todos los registros
# en una sola instrucción (vectorización), sin necesidad de bucles innecesarios.

Primeros totales calculados: [  110. 10827.  3912.  2556.  5142.]
Shape de la matriz de totales: (4883,)


---
## Parte 5 — Análisis con NumPy

In [91]:
# Cálculos con funciones de NumPy
suma_total    = np.sum(totales)
promedio      = np.mean(totales)
venta_max_np  = np.max(totales)
ventas_sup_1000 = np.sum(totales > 1000)

print(f"Suma total de ventas:           $ {suma_total:,.2f}")
print(f"Promedio de ventas:             $ {promedio:,.2f}")
print(f"Venta máxima:                   $ {venta_max_np:,.2f}")
print(f"Cantidad de ventas > $ 1,000:    {ventas_sup_1000}")

Suma total de ventas:           $ 24,477,978.00
Promedio de ventas:             $ 5,012.90
Venta máxima:                   $ 17,955.00
Cantidad de ventas > $ 1,000:    4136


### Respuestas — Parte 5

**¿Qué ventajas observas al usar NumPy?**

---
NumPy es más legible dentro del código, y además, según lo que entiendo es significativamente más rápido para cálculos, pues esa es su función. También permite operar en un contexto de álgebra lineal, por lo que se puede entender mejor dentro del concepto de la programación.

--- 

**¿Qué significa "vectorización"?**

Vectorización significa aplicar una operación matemática **simultáneamente a todos los elementos** de una matriz, en lugar de recorrerlos uno por uno con un bucle.


---

**¿Qué hace la expresión `data[:, 0]`?**

- `:` selecciona **todas las filas**
- `0` selecciona la **columna de índice 0** (precios)

El resultado es una matriz con todos los valores de la primera columna.

---
## Parte 6 — Interpretación de Resultados

In [99]:
# Revisión de distribución de totales para evaluar el promedio
print("Distribución de 'total' ")
print(df_clean['total'].describe())

# Revisión de sesgo: comparar media vs mediana
media   = df_clean['total'].mean()
mediana = df_clean['total'].median()
print(f"\nMedia:   $ {media:,.2f}")
print(f"Mediana: $ {mediana:,.2f}")
print(f"Sesgo:   {'positivo (media > mediana)' if media > mediana else 'negativo'}")

Distribución de 'total' 
count     4883.000000
mean      5012.897399
std       4122.959359
min         10.000000
25%       1645.000000
50%       3864.000000
75%       7460.500000
max      17955.000000
Name: total, dtype: float64

Media:   $ 5,012.90
Mediana: $ 3,864.00
Sesgo:   positivo (media > mediana)


In [100]:
# Revisar si quedan valores sospechosos
print("Precios extremos luego de limpieza")
print(df_clean.nlargest(5, 'precio')[['producto', 'precio', 'cantidad', 'total']])

print("\nTotales extremos")
print(df_clean.nlargest(5, 'total')[['producto', 'precio', 'cantidad', 'total']])

Precios extremos luego de limpieza
     producto  precio  cantidad    total
196   Celular  1997.0       5.0   9985.0
227     Mouse  1996.0       4.0   7984.0
2808   Laptop  1996.0       7.0  13972.0
2949  Teclado  1996.0       3.0   5988.0
3393  Monitor  1996.0       4.0   7984.0

Totales extremos
     producto  precio  cantidad    total
1378    Mouse  1995.0       9.0  17955.0
347   Teclado  1989.0       9.0  17901.0
3079  Teclado  1987.0       9.0  17883.0
294   Monitor  1986.0       9.0  17874.0
206    Laptop  1981.0       9.0  17829.0


### Respuestas — Parte 6

**¿Los resultados obtenidos tienen sentido?**

Sí. Tras la limpieza, los precios oscilan entre $10 y $1.999, y las cantidades entre 1 y 10 unidades, lo cual es consistente con un catálogo de productos tecnológicos (mouse, teclado, celular, laptop, monitor). El total de ventas cercano a **$24,5 millones** para ~4.883 transacciones implica un ticket promedio de ~$5.000, razonable para ese tipo de productos.

---

**¿Detectaste valores sospechosos?**

Sí, se detectaron dos tipos:
1. **`precio == 999999`**: Valor centinela (20 registros) que indica error de carga. Se eliminaron.
2. **`cantidad = 'three'`**: Dato textual en columna numérica, posiblemente ingresado manualmente. Se corrigió a `3`.

---

**¿El promedio representa correctamente los datos?**

La media y la mediana de `total` son muy cercanas entre sí, lo que indica que la distribución es relativamente simétrica y el promedio **sí es representativo** en este caso. Si hubiera outliers extremos sin limpiar (como el `999999`), el promedio se distorsionaría fuertemente y dejaría de ser un buen indicador central.

---

**¿Qué decisiones tomarías si esta fuera información real de negocio?**

1. **Validación en origen**: Implementar controles de calidad en el sistema de ingreso de datos para evitar que valores como `'three'` o `999999` ingresen al sistema.
2. **Análisis por período**: Cruzar ventas con fechas para identificar estacionalidad o picos de demanda.
3. **Segmentación por país**: Colombia tiene el mayor volumen; valdría la pena profundizar en qué productos impulsan esa diferencia.
4. **Revisión de Mouse**: Lidera en ingresos totales. Analizar si es por volumen de unidades o por precio promedio.